# Word embeddings and alignment

I compare cosine similarity before and after lexicon-based retrofitting, then recover a synthetic rotation with orthogonal Procrustes. The small corpus keeps the calculations easy to inspect.

In [ ]:
from gensim.models import Word2Vec
import numpy as np
from nlp_portfolio.embeddings import cosine, align_orthogonal, retrofit

corpus = [
    'doctor nurse hospital patient treatment',
    'doctor hospital illness patient',
    'research scientist experiment laboratory discovery',
    'scientist research laboratory data',
    'computer algorithm data model research',
    'nurse patient care hospital',
] * 30
sentences = [line.split() for line in corpus]
word2vec = Word2Vec(sentences, vector_size=30, window=3, min_count=1, workers=1, seed=42, epochs=40)
print(word2vec.wv.most_similar('doctor', topn=4))

In [ ]:
before = {'doctor': word2vec.wv['doctor'], 'nurse': word2vec.wv['nurse'], 'hospital': word2vec.wv['hospital']}
lexicon = {'doctor': ['nurse'], 'nurse': ['doctor'], 'hospital': []}
after = retrofit(before, lexicon, iterations=10)
print('Before:', round(cosine(before['doctor'], before['nurse']), 3))
print('After: ', round(cosine(after['doctor'], after['nurse']), 3))

In [ ]:
rng = np.random.default_rng(42)
source = rng.normal(size=(30, 8))
rotation, _ = np.linalg.qr(rng.normal(size=(8, 8)))
target = source @ rotation
estimated = align_orthogonal(source, target)
print('Alignment mean-squared error:', np.mean((source @ estimated - target) ** 2))

The vectors trained here are too small for linguistic conclusions. Comparing embeddings from different corpora requires alignment using shared words; independent Word2Vec spaces have arbitrary rotations.